In [11]:
# EXERCISE 1: WEB SERVER ACCESS LOG ANALYSIS
from pathlib import Path
import json
import shutil

LAST_NAME = "Agustin"
STUDENT_ID = "TUPM-26-1937"

SEED_DIGIT = int(STUDENT_ID[-1])
ID_SUM = sum(int(d) for d in STUDENT_ID if d.isdigit())
NAME_LENGTH = len(LAST_NAME)

print("Seed:", SEED_DIGIT)
print("ID Sum:", ID_SUM)
print("Name Length:", NAME_LENGTH)

WORK_DIR = Path(".")
DATA_DIR = WORK_DIR / "data"
OUTPUT_DIR = WORK_DIR / "output"
BACKUP_DIR = WORK_DIR / "backup"

DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)
BACKUP_DIR.mkdir(exist_ok=True)

print("Workspace:", WORK_DIR.resolve())
print("Data folder exists:", DATA_DIR.exists())
print("Output folder exists:", OUTPUT_DIR.exists())
print("Backup folder exists:", BACKUP_DIR.exists())

RECORDS_TO_PROCESS = 1000 + ID_SUM
TARGET_CLASS = (SEED_DIGIT % 4) + 2

print("Records to process:", RECORDS_TO_PROCESS)
print("Target status class:", str(TARGET_CLASS) + "xx")

dataset_path = DATA_DIR / "weblog.csv"
filtered_path = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_weblog_filtered.csv"

records_processed = 0
invalid_records = 0 
records_in_target = 0

status_counts = {}
target_status_counts = {}
url_counts = {}

if not dataset_path.exists():
    print("Dataset not found:", dataset_path)
else:
    with open(dataset_path, "r", encoding="utf-8-sig") as source:
        header = source.readline().strip().split(",")
        header = [name.strip() for name in header]

        ip_index = header.index("IP")
        time_index = header.index("Time")
        url_index = header.index("URL")
        status_index = header.index("Staus")

        with open(filtered_path, "w", encoding="utf-8") as output:
            output.write("IP,Time,URL,Status\n")

            for line in source:
                if records_processed >= RECORDS_TO_PROCESS:
                    break

                line = line.strip()
                if line == "":
                    continue

                records_processed += 1
                fields = line.split(",")

                if len(fields) != len(header):
                    invalid_records += 1
                    continue

                ip = fields[ip_index].strip()
                timestamp = fields[time_index].strip()
                url = fields[url_index].strip()
                status = fields[status_index].strip()

                if ip == "" or timestamp == "" or url == "":
                    invalid_records += 1
                    continue
                if len(status) != 3 or not status.isdigit():
                    invalid_records += 1
                    continue

                status_counts[status] = status_counts.get(status, 0) + 1

                if int(status[0]) == TARGET_CLASS:
                    records_in_target += 1
                    target_status_counts[status] = target_status_counts.get(status, 0) + 1
                    url_counts[url] = url_counts.get(url, 0) + 1
                    output.write(f"{ip},{timestamp},{url},{status}\n")

print("Records processed:", records_processed)
print("Invalid records skipped:", invalid_records)
print("Records in target class:", records_in_target)
print("Filtered CSV created:", filtered_path)

print("Count per status code (all valid records processed):")
for code in sorted(status_counts):
    print(" ", code, ":", status_counts[code])

print("Count per status code (target class only):")
for code in sorted(target_status_counts):
    print(" ", code, ":", target_status_counts[code])

remaining = list(url_counts.items())
top_urls = []

for _ in range(5):
    if len(remaining) == 0:
        break
    best = 0
    for i in range(1, len(remaining)):
        if remaining[i][1] > remaining[best][1]:
            best = i
    top_urls.append(remaining.pop(best))

print("Five most requested URLs in the target class:")
if len(top_urls) == 0:
    print("  No records in the target class.")
for rank, (url, count) in enumerate(top_urls, 1):
    print(f"  {rank}. {url} -> {count}")

json_path = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_weblog_summary.json"

top_url_list = []
for url, count in top_urls:
    top_url_list.append({"url": url, "requests": count})

summary = {
    "last_name": LAST_NAME,
    "student_id": STUDENT_ID,
    "seed_digit": SEED_DIGIT,
    "id_sum": ID_SUM,
    "records_processed": records_processed,
    "invalid_records": invalid_records,
    "target_status_class": str(TARGET_CLASS) + "xx",
    "records_in_target_class": records_in_target,
    "count_per_status_code_all": status_counts,
    "count_per_status_code_target_class": target_status_counts,
    "five_most_requested_urls": top_url_list,
    "filtered_csv_file": filtered_path.name,
}

with open(json_path, "w", encoding="utf-8") as file:
    json.dump(summary, file, indent=4)

print("JSON summary created:", json_path)

backup_path = BACKUP_DIR / json_path.name
shutil.copy2(json_path, backup_path)

backup_ok = backup_path.exists() and backup_path.stat().st_size == json_path.stat().st_size

print("Original:", json_path)
print("Backup:", backup_path)
print("Backup exists:", backup_path.exists())
print("Backup verification:", "VERIFIED" if backup_ok else "CHECK")

print("ASSESSMENT DATA - EXERCISE 1")
print("-" * 40)
print("Records processed      :", records_processed)
print("Target status class    :", str(TARGET_CLASS) + "xx")
print("Records in target class:", records_in_target)
print("Count per status code  :", status_counts)
print("Five most requested URLs:")
for url, count in top_urls:
    print("   ", url, "->", count)
print("Filtered CSV file      :", filtered_path.name)
print("JSON summary file      :", json_path.name)
print("Backup verification    :", "VERIFIED" if backup_ok else "CHECK")

Seed: 7
ID Sum: 28
Name Length: 7
Workspace: C:\Users\Lenar Agustin\Documents\Lab6_File_Handling_Additional_Exercises
Data folder exists: True
Output folder exists: True
Backup folder exists: True
Records to process: 1028
Target status class: 5xx
Records processed: 1028
Invalid records skipped: 0
Records in target class: 0
Filtered CSV created: output\Agustin_TUPM-26-1937_weblog_filtered.csv
Count per status code (all valid records processed):
  200 : 850
  206 : 8
  302 : 144
  304 : 21
  404 : 5
Count per status code (target class only):
Five most requested URLs in the target class:
  No records in the target class.
JSON summary created: output\Agustin_TUPM-26-1937_weblog_summary.json
Original: output\Agustin_TUPM-26-1937_weblog_summary.json
Backup: backup\Agustin_TUPM-26-1937_weblog_summary.json
Backup exists: True
Backup verification: VERIFIED
ASSESSMENT DATA - EXERCISE 1
----------------------------------------
Records processed      : 1028
Target status class    : 5xx
Records in 

In [12]:
# EXERCISE 2: MOBILE DEVICE USAGE PROFILING
from pathlib import Path
import json
import shutil

LAST_NAME = "Agustin"
STUDENT_ID = "TUPM-26-1937"

SEED_DIGIT = int(STUDENT_ID[-1])
ID_SUM = sum(int(d) for d in STUDENT_ID if d.isdigit())
NAME_LENGTH = len(LAST_NAME)

print("Seed:", SEED_DIGIT)
print("ID Sum:", ID_SUM)
print("Name Length:", NAME_LENGTH)

WORK_DIR = Path(".")
DATA_DIR = WORK_DIR / "data"
OUTPUT_DIR = WORK_DIR / "output"
BACKUP_DIR = WORK_DIR / "backup"

DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)
BACKUP_DIR.mkdir(exist_ok=True)

print("Workspace:", WORK_DIR.resolve())
print("Data folder exists:", DATA_DIR.exists())
print("Output folder exists:", OUTPUT_DIR.exists())
print("Backup folder exists:", BACKUP_DIR.exists())

if SEED_DIGIT % 2 == 0:
    TARGET_OS = "Android"
else:
    TARGET_OS = "iOS"

TARGET_CLASS = (SEED_DIGIT % 5) + 1

print("Target operating system:", TARGET_OS)
print("Target behavior class:", TARGET_CLASS)

dataset_path = DATA_DIR / "user_behavior_dataset.csv"
export_path = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_selected_users.csv"

total_records = 0
invalid_records = 0
os_matches = 0
selected_count = 0
total_battery = 0.0
total_data = 0.0

if not dataset_path.exists():
    print("Dataset not found:", dataset_path)
else:
    with open(dataset_path, "r", encoding="utf-8-sig") as source:
        header = source.readline().strip().split(",")
        header = [name.strip() for name in header]

        id_index = header.index("User ID")
        model_index = header.index("Device Model")
        os_index = header.index("Operating System")
        battery_index = header.index("Battery Drain (mAh/day)")
        data_index = header.index("Data Usage (MB/day)")
        class_index = header.index("User Behavior Class")

        with open(export_path, "w", encoding="utf-8") as output:
            output.write("User ID,Device Model,Operating System,Battery Drain (mAh/day),Data Usage (MB/day)\n")

            for line in source:
                line = line.strip()
                if line == "":
                    continue

                total_records += 1
                fields = line.split(",")

                if len(fields) != len(header):
                    invalid_records += 1
                    continue

                user_id = fields[id_index].strip()
                model = fields[model_index].strip()
                os_name = fields[os_index].strip()

                try:
                    battery = float(fields[battery_index])
                    data_usage = float(fields[data_index])
                    behavior_class = int(fields[class_index])
                except ValueError:
                    invalid_records += 1
                    continue

                if os_name == TARGET_OS:
                    os_matches += 1

                    if behavior_class == TARGET_CLASS:
                        selected_count += 1
                        total_battery += battery
                        total_data += data_usage
                        output.write(f"{user_id},{model},{os_name},{fields[battery_index].strip()},{fields[data_index].strip()}\n")

print("Total records in dataset:", total_records)
print("Invalid records skipped:", invalid_records)
print("Records matching operating system:", os_matches)
print("Records in target behavior class:", selected_count)
print("Exported CSV created:", export_path)

if selected_count > 0:
    average_battery = total_battery / selected_count
    average_data = total_data / selected_count
else:
    average_battery = 0.0
    average_data = 0.0
    print("No matching records found. Averages set to 0.")

print("Average battery drain (mAh/day):", round(average_battery, 2))
print("Average data usage (MB/day):", round(average_data, 2))

exported_rows = 0
with open(export_path, "r", encoding="utf-8") as file:
    next(file)                       # skip the header line
    for line in file:
        if line.strip() != "":
            exported_rows += 1

count_ok = exported_rows == selected_count

print("Exported rows:", exported_rows)
print("Selected records:", selected_count)
print("Count verification:", "VERIFIED" if count_ok else "CHECK")

json_path = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_usage_summary.json"

summary = {
    "last_name": LAST_NAME,
    "student_id": STUDENT_ID,
    "seed_digit": SEED_DIGIT,
    "target_operating_system": TARGET_OS,
    "target_behavior_class": TARGET_CLASS,
    "total_records": total_records,
    "invalid_records": invalid_records,
    "records_matching_os": os_matches,
    "records_in_target_class": selected_count,
    "average_battery_drain_mah_day": round(average_battery, 2),
    "average_data_usage_mb_day": round(average_data, 2),
    "exported_row_count": exported_rows,
    "count_verification": "VERIFIED" if count_ok else "CHECK",
    "exported_csv_file": export_path.name,
}

with open(json_path, "w", encoding="utf-8") as file:
    json.dump(summary, file, indent=4)

print("JSON summary created:", json_path)

backup_path = BACKUP_DIR / export_path.name
shutil.copy2(export_path, backup_path)

backup_size = backup_path.stat().st_size

print("Original:", export_path)
print("Backup:", backup_path)
print("Backup exists:", backup_path.exists())
print("Backup file size:", backup_size, "bytes")

print("ASSESSMENT DATA - EXERCISE 2")
print("-" * 40)
print("Operating system and behavior class:", TARGET_OS, "/ Class", TARGET_CLASS)
print("Total records in dataset           :", total_records)
print("Records matching operating system  :", os_matches)
print("Records in target behavior class   :", selected_count)
print("Average battery drain (mAh/day)    :", round(average_battery, 2))
print("Average data usage (MB/day)        :", round(average_data, 2))
print("Count verification                 :", "VERIFIED" if count_ok else "CHECK")
print("Exported CSV file                  :", export_path.name)
print("JSON summary file                  :", json_path.name)
print("Backup file and size               :", backup_path.name, "-", backup_size, "bytes")

Seed: 7
ID Sum: 28
Name Length: 7
Workspace: C:\Users\Lenar Agustin\Documents\Lab6_File_Handling_Additional_Exercises
Data folder exists: True
Output folder exists: True
Backup folder exists: True
Target operating system: iOS
Target behavior class: 3
Total records in dataset: 700
Invalid records skipped: 0
Records matching operating system: 146
Records in target behavior class: 32
Exported CSV created: output\Agustin_TUPM-26-1937_selected_users.csv
Average battery drain (mAh/day): 1492.03
Average data usage (MB/day): 828.09
Exported rows: 32
Selected records: 32
Count verification: VERIFIED
JSON summary created: output\Agustin_TUPM-26-1937_usage_summary.json
Original: output\Agustin_TUPM-26-1937_selected_users.csv
Backup: backup\Agustin_TUPM-26-1937_selected_users.csv
Backup exists: True
Backup file size: 973 bytes
ASSESSMENT DATA - EXERCISE 2
----------------------------------------
Operating system and behavior class: iOS / Class 3
Total records in dataset           : 700
Records mat

In [13]:
# EXERCISE 3: CYBERSECURITY INCIDENT REPORT
from pathlib import Path
import json
import shutil

LAST_NAME = "Agustin"
STUDENT_ID = "TUPM-26-1937"

SEED_DIGIT = int(STUDENT_ID[-1])
ID_SUM = sum(int(d) for d in STUDENT_ID if d.isdigit())
NAME_LENGTH = len(LAST_NAME)

print("Seed:", SEED_DIGIT)
print("ID Sum:", ID_SUM)
print("Name Length:", NAME_LENGTH)

WORK_DIR = Path(".")
DATA_DIR = WORK_DIR / "data"
OUTPUT_DIR = WORK_DIR / "output"
BACKUP_DIR = WORK_DIR / "backup"

DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)
BACKUP_DIR.mkdir(exist_ok=True)

print("Workspace:", WORK_DIR.resolve())
print("Data folder exists:", DATA_DIR.exists())
print("Output folder exists:", OUTPUT_DIR.exists())
print("Backup folder exists:", BACKUP_DIR.exists())

TARGET_YEAR = 2015 + SEED_DIGIT

print("Target year:", TARGET_YEAR)

dataset_path = DATA_DIR / "Global_Cybersecurity_Threats_2015-2024.csv"
filtered_path = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_incidents_filtered.csv"

total_records = 0
invalid_records = 0
selected_incidents = []
header = []

if not dataset_path.exists():
    print("Dataset not found:", dataset_path)
else:
    with open(dataset_path, "r", encoding="utf-8-sig") as source:
        header_line = source.readline().strip()
        header = [name.strip() for name in header_line.split(",")]

        year_index = header.index("Year")
        attack_index = header.index("Attack Type")
        loss_index = header.index("Financial Loss (in Million $)")

        with open(filtered_path, "w", encoding="utf-8") as output:
            output.write(header_line + "\n")

            for line in source:
                line = line.strip()
                if line == "":
                    continue

                total_records += 1
                fields = line.split(",")

                if len(fields) != len(header):
                    invalid_records += 1
                    continue

                try:
                    year = int(fields[year_index])
                except ValueError:
                    invalid_records += 1
                    continue

                if year == TARGET_YEAR:
                    output.write(line + "\n")

                    record = {}
                    for name, value in zip(header, fields):
                        record[name] = value.strip()
                    selected_incidents.append(record)

incidents_in_year = len(selected_incidents)

print("Total records in dataset:", total_records)
print("Invalid records skipped:", invalid_records)
print("Incidents in target year:", incidents_in_year)
print("Filtered CSV created:", filtered_path)

attack_counts = {}

for record in selected_incidents:
    attack = record["Attack Type"]
    attack_counts[attack] = attack_counts.get(attack, 0) + 1

print("Count per attack type:")
if incidents_in_year == 0:
    print("  No incidents found for the target year.")
for attack in sorted(attack_counts):
    print(" ", attack, ":", attack_counts[attack])

total_loss = 0.0
valid_loss_count = 0
invalid_loss_count = 0
valid_losses = []

for position, record in enumerate(selected_incidents):
    try:
        loss = float(record["Financial Loss (in Million $)"])
    except ValueError:
        invalid_loss_count += 1
        continue

    total_loss += loss
    valid_loss_count += 1
    valid_losses.append((position, loss, record))

if valid_loss_count > 0:
    average_loss = total_loss / valid_loss_count
else:
    average_loss = 0.0
    print("No valid financial values found. Average set to 0.")

print("Incidents with valid loss values:", valid_loss_count)
print("Incidents with invalid loss values:", invalid_loss_count)
print("Total financial loss (USD millions):", round(total_loss, 2))
print("Average financial loss (USD millions):", round(average_loss, 2))

remaining = list(valid_losses)
top_five = []

for _ in range(5):
    if len(remaining) == 0:
        break
    best = 0
    for i in range(1, len(remaining)):
        if remaining[i][1] > remaining[best][1]:
            best = i
    top_five.append(remaining.pop(best))

print("Five costliest incidents:")
if len(top_five) == 0:
    print("  No incidents to rank.")
for rank, (position, loss, record) in enumerate(top_five, 1):
    print(f"  {rank}. {record['Attack Type']} | {record['Country']} | {loss} million USD")

json_path = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_incident_summary.json"

top_five_records = []
for position, loss, record in top_five:
    top_five_records.append(record)

summary = {
    "last_name": LAST_NAME,
    "student_id": STUDENT_ID,
    "seed_digit": SEED_DIGIT,
    "target_year": TARGET_YEAR,
    "total_records": total_records,
    "invalid_records": invalid_records,
    "incidents_in_target_year": incidents_in_year,
    "count_per_attack_type": attack_counts,
    "invalid_financial_values": invalid_loss_count,
    "total_financial_loss_usd_millions": round(total_loss, 2),
    "average_financial_loss_usd_millions": round(average_loss, 2),
    "five_costliest_incidents": top_five_records,
    "filtered_csv_file": filtered_path.name,
}

with open(json_path, "w", encoding="utf-8") as file:
    json.dump(summary, file, indent=4)

print("JSON summary created:", json_path)

backup_path = BACKUP_DIR / json_path.name
shutil.copy2(json_path, backup_path)

backup_ok = backup_path.exists() and backup_path.stat().st_size == json_path.stat().st_size

print("Original:", json_path)
print("Backup:", backup_path)
print("Backup exists:", backup_path.exists())
print("Backup verification:", "VERIFIED" if backup_ok else "CHECK")

report_path = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_incident_report.txt"

with open(report_path, "w", encoding="utf-8") as report:
    report.write("CYBERSECURITY INCIDENT REPORT\n")
    report.write("-" * 40 + "\n")
    report.write(f"Last Name: {LAST_NAME}\n")
    report.write(f"Student ID: {STUDENT_ID}\n")
    report.write(f"Target Year: {TARGET_YEAR}\n")
    report.write(f"Total Records in Dataset: {total_records}\n")
    report.write(f"Incidents in Target Year: {incidents_in_year}\n")
    report.write("\nINCIDENT COUNT PER ATTACK TYPE\n")
    for attack in sorted(attack_counts):
        report.write(f"  {attack}: {attack_counts[attack]}\n")
    report.write("\nFINANCIAL LOSS (USD MILLIONS)\n")
    report.write(f"  Total: {total_loss:.2f}\n")
    report.write(f"  Average: {average_loss:.2f}\n")
    report.write(f"  Invalid values skipped: {invalid_loss_count}\n")
    report.write("\nFIVE COSTLIEST INCIDENTS\n")
    for rank, (position, loss, record) in enumerate(top_five, 1):
        report.write(f"  {rank}. {record['Attack Type']} | {record['Country']} | {record['Year']} | {loss}\n")
    report.write("\nOUTPUT FILES\n")
    report.write(f"  Filtered CSV: {filtered_path.name}\n")
    report.write(f"  JSON Summary: {json_path.name}\n")
    report.write(f"  JSON Backup: {backup_path.name} ({'VERIFIED' if backup_ok else 'CHECK'})\n")
    report.write(f"  Report: {report_path.name}\n")

print("Report created:", report_path)

print("ASSESSMENT DATA - EXERCISE 3")
print("-" * 40)
print("Target year                       :", TARGET_YEAR)
print("Total records in dataset          :", total_records)
print("Incidents in target year          :", incidents_in_year)
print("Count per attack type             :", attack_counts)
print("Total financial loss (USD millions):", round(total_loss, 2))
print("Average financial loss (USD millions):", round(average_loss, 2))
print("Five costliest incidents:")
for rank, (position, loss, record) in enumerate(top_five, 1):
    print(f"   {rank}. {record['Attack Type']} | {record['Country']} | {loss}")
print("Filtered CSV file                 :", filtered_path.name)
print("JSON summary file                 :", json_path.name)
print("Backup verification               :", "VERIFIED" if backup_ok else "CHECK")
print("Report file                       :", report_path.name)

Seed: 7
ID Sum: 28
Name Length: 7
Workspace: C:\Users\Lenar Agustin\Documents\Lab6_File_Handling_Additional_Exercises
Data folder exists: True
Output folder exists: True
Backup folder exists: True
Target year: 2022
Total records in dataset: 3000
Invalid records skipped: 0
Incidents in target year: 318
Filtered CSV created: output\Agustin_TUPM-26-1937_incidents_filtered.csv
Count per attack type:
  DDoS : 54
  Malware : 64
  Man-in-the-Middle : 41
  Phishing : 57
  Ransomware : 56
  SQL Injection : 46
Incidents with valid loss values: 318
Incidents with invalid loss values: 0
Total financial loss (USD millions): 15870.86
Average financial loss (USD millions): 49.91
Five costliest incidents:
  1. Phishing | Japan | 98.47 million USD
  2. SQL Injection | USA | 98.46 million USD
  3. Phishing | Brazil | 98.06 million USD
  4. Phishing | UK | 97.56 million USD
  5. Malware | Japan | 97.32 million USD
JSON summary created: output\Agustin_TUPM-26-1937_incident_summary.json
Original: output\Ag